# SonarVision Demo

Depth sounder → underwater video prediction with self-supervised learning.

This notebook walks through: loading depth data, preprocessing, running inference, and visualizing results.

In [ ]:
import sys, os
sys.path.insert(0, '..')
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from sonar_vision.config import load_config
from sonar_vision.water.physics import UnderwaterPhysics
from sonar_vision.encoder import DepthEncoder
from sonar_vision.decoder import VideoDecoder

print('SonarVision modules loaded successfully')

## 1. Load Configuration

In [ ]:
config = load_config('../configs/default.yaml')
print(f'Water type: {config.water.water_type}')
print(f'Encoder dim: {config.encoder.latent_dim}')
print(f'Decoder fps: {config.decoder.fps}')

## 2. Generate Synthetic Depth Data

In [ ]:
t = np.linspace(0, 10, 200)
x = np.linspace(0, 50, 64)
T, X = np.meshgrid(t, x, indexing='ij')

depth_data = 5 + 3 * np.sin(0.5 * T) * np.exp(-0.1 * X) + 0.5 * np.random.randn(200, 64)
depth_data = depth_data.astype(np.float32)

print(f'Depth data shape: {depth_data.shape}')
print(f'Depth range: {depth_data.min():.1f}m - {depth_data.max():.1f}m')

## 3. Apply Underwater Physics

In [ ]:
physics = UnderwaterPhysics(config.water)
attenuated = physics.simulate(depth_data)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
im1 = ax1.imshow(depth_data.T, aspect='auto', cmap='viridis')
ax1.set_title('Raw Depth Data')
plt.colorbar(im1, ax=ax1, label='Depth (m)')
im2 = ax2.imshow(attenuated.T, aspect='auto', cmap='inferno')
ax2.set_title('After Physics Simulation')
plt.colorbar(im2, ax=ax2, label='Attenuated Signal')
plt.tight_layout()
plt.show()

## 4. Encode Depth Data

In [ ]:
encoder = DepthEncoder(config.encoder)
latent = encoder.encode(attenuated)
print(f'Latent representation shape: {latent.shape}')

## 5. Decode to Video Frames

In [ ]:
decoder = VideoDecoder(config.decoder)
frames = decoder.decode(latent)
print(f'Generated {len(frames)} video frames')
print(f'Frame shape: {frames[0].shape if frames else "N/A"}')

## 6. Visualize Results

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 6))
for i, ax in enumerate(axes.flat):
    if i < len(frames):
        ax.imshow(frames[i], cmap='plasma')
        ax.set_title(f'Frame {i+1}')
        ax.axis('off')
plt.tight_layout()
plt.show()

## 7. Performance Metrics

In [ ]:
import time
start = time.perf_counter()
_ = physics.simulate(depth_data)
_ = encoder.encode(attenuated)
_ = decoder.decode(latent)
elapsed = (time.perf_counter() - start) * 1000
print(f'End-to-end inference: {elapsed:.1f}ms')
print(f'Effective frame rate: {len(frames) / (elapsed / 1000):.1f} fps')